# Final-B 17-class — Evaluate and register runs

Re-evaluate all 30 finalized checkpoints on validation and test, export prediction/report CSVs, and create one immutable timestamped registry. Each registry row represents one training run; every other report table carries lineage back to those `run_id` values.

In [1]:
from pathlib import Path
import sys

if Path('/content').exists():
    ROOT = Path('/content/woodbench')
else:
    cwd = Path.cwd()
    candidates = [cwd, *cwd.parents]
    ROOT = next(
        (candidate for candidate in candidates if (candidate / 'src').exists() or (candidate / 'pyproject.toml').exists()),
        cwd,
    )

sys.path.insert(0, str(ROOT))
print(f'[ROOT] {ROOT}')

[ROOT] /workspace/woodbench


In [2]:
if Path('/content').exists():
    !pip install -q -r {ROOT}/requirements.txt

    DATA_DIR = ROOT / "data" / "task-B-group-split"
    if not DATA_DIR.exists():
        from huggingface_hub import snapshot_download

        snapshot_download(
            repo_id="ny0g507h3p/wood-task-B",
            repo_type="dataset",
            local_dir=str(DATA_DIR),
            ignore_patterns=["*.md", "README*"],
        )

In [3]:
import gc
import hashlib
import json
import os
import platform
import subprocess
from contextlib import nullcontext
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, cohen_kappa_score,
    confusion_matrix, log_loss, matthews_corrcoef,
    precision_recall_fscore_support, top_k_accuracy_score,
)
from torch.utils.data import DataLoader

from src.data import FlatConiferDataset
from src.data.transforms import get_val_transforms
from src.eval import save_confusion_matrix
from src.models import create_model

DATA_DIR = ROOT / "data" / "task-B-group-split"
EXPERIMENT_DIR = ROOT / "data" / "experiments" / "Task-B-new" / "final"
OUTPUT_DIR = ROOT / "docs" / "reports" / "experiments" / "phase_1b" / "final_b_new_recomputed"
EVALUATION_NOTEBOOK = "notebooks/phase 1/task-B-17-new/06_evaluate_and_register_runs.ipynb"
REGISTRY_NAME = "task-b-17class"
REGISTRY_CREATED_AT = datetime.now(timezone.utc)
REGISTRY_TIMESTAMP = REGISTRY_CREATED_AT.strftime("%Y%m%dT%H%M%SZ")
RUN_OUTPUT_DIR = OUTPUT_DIR / REGISTRY_TIMESTAMP
REGISTRY_PATH = RUN_OUTPUT_DIR / f"{REGISTRY_NAME}-{REGISTRY_TIMESTAMP}-registry.csv"
PREDICTION_DIR = RUN_OUTPUT_DIR / "predictions"

BACKBONES = ["resnet50", "efficientnet_b3", "mobilenetv3_large", "convnext_tiny", "swin_tiny"]
MAIN_SEEDS = [42, 123, 456]
ABLATIONS = ["taskB_90rotation", "taskB_label_smoothing_0.0", "taskB_label_smoothing_0.2"]
TRAINING_NOTEBOOKS = {
    "resnet50": "notebooks/phase 1/task-B-17-new/01_train_resnet50.ipynb",
    "efficientnet_b3": "notebooks/phase 1/task-B-17-new/02_train_efficientnet_b3.ipynb",
    "mobilenetv3_large": "notebooks/phase 1/task-B-17-new/03_train_mobilenetv3_large.ipynb",
    "convnext_tiny": "notebooks/phase 1/task-B-17-new/04_train_convnext_tiny.ipynb",
    "swin_tiny": "notebooks/phase 1/task-B-17-new/05_train_swin_tiny.ipynb",
}
BACKBONE_BATCH_SIZES = {"resnet50": 64, "efficientnet_b3": 32, "mobilenetv3_large": 64, "convnext_tiny": 32, "swin_tiny": 32}
EXPERIMENT_DEFAULTS = {
    "main": {"label_smoothing": 0.1, "use_90rotation": False},
    "taskB_90rotation": {"label_smoothing": 0.1, "use_90rotation": True},
    "taskB_label_smoothing_0.0": {"label_smoothing": 0.0, "use_90rotation": False},
    "taskB_label_smoothing_0.2": {"label_smoothing": 0.2, "use_90rotation": False},
}
EXCLUDED_CLASSES = {"Calocedrus sp", "Cupressus funebris (Cupressus pendula)", "Pinus sp"}
EXPECTED_SPLIT_COUNTS = {"train": 3365, "val": 903, "test": 532}
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png"}
IMAGE_SIZE = 224
DATA_LOADING_BATCH_SIZE = 256
MAX_EVAL_BATCH_SIZE = 1024
MIN_EVAL_BATCH_SIZE = 16
NUM_WORKERS = min(8, os.cpu_count() or 2)
PREFETCH_FACTOR = 4
GPU_CACHE_RESERVE_GB = 2.0
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
CACHE_EVAL_ON_GPU = DEVICE.type == "cuda"

if DEVICE.type == "cuda":
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.set_float32_matmul_precision("high")

print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(DEVICE)}")
    total_vram_gb = torch.cuda.get_device_properties(DEVICE).total_memory / 2**30
    print(f"VRAM: {total_vram_gb:.1f} GiB | AMP: {USE_AMP} | TF32: enabled")
print(f"Workers: {NUM_WORKERS} | preload batch: {DATA_LOADING_BATCH_SIZE} | max eval batch: {MAX_EVAL_BATCH_SIZE}")
print(f"Checkpoints: {EXPERIMENT_DIR}")
print(f"Run output: {RUN_OUTPUT_DIR}")
print(f"Registry: {REGISTRY_PATH.name}")

Device: cuda
GPU: NVIDIA GeForce RTX 4090
VRAM: 23.5 GiB | AMP: True | TF32: enabled
Workers: 8 | preload batch: 256 | max eval batch: 1024
Checkpoints: /workspace/woodbench/data/experiments/Task-B-new/final
Run output: /workspace/woodbench/docs/reports/experiments/phase_1b/final_b_new_recomputed/20260819T043733Z
Registry: task-b-17class-20260819T043733Z-registry.csv


In [4]:
def relative_to_root(path: Path) -> str:
    return Path(path).resolve().relative_to(ROOT.resolve()).as_posix()


def current_git_commit(root: Path) -> str:
    try:
        return subprocess.run(
            ["git", "rev-parse", "HEAD"], cwd=root, check=True,
            capture_output=True, text=True,
        ).stdout.strip()
    except (FileNotFoundError, subprocess.CalledProcessError):
        return "unknown"


def validate_dataset(data_dir: Path) -> tuple[list[str], dict[str, int], str]:
    split_classes, split_counts, manifest_rows = {}, {}, []
    for split, expected_count in EXPECTED_SPLIT_COUNTS.items():
        split_dir = data_dir / split
        if not split_dir.exists():
            raise FileNotFoundError(f"Missing Final-B split: {split_dir}")
        classes = sorted(path.name for path in split_dir.iterdir() if path.is_dir())
        image_paths = sorted(
            path for path in split_dir.rglob("*")
            if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
        )
        actual_count = len(image_paths)
        assert len(classes) == 17, f"{split}: expected 17 classes, got {len(classes)}"
        assert EXCLUDED_CLASSES.isdisjoint(classes), f"{split}: excluded class remains"
        assert actual_count == expected_count, f"{split}: expected {expected_count} images, got {actual_count}"
        split_classes[split], split_counts[split] = classes, actual_count
        for path in image_paths:
            with path.open("rb") as image_file:
                image_digest = hashlib.file_digest(image_file, "sha256").hexdigest()
            manifest_rows.append(f"{relative_to_root(path)}\t{image_digest}")

    reference = split_classes["train"]
    for split in ["val", "test"]:
        assert split_classes[split] == reference, f"{split}: class mapping differs from train"
    digest = hashlib.sha256("\n".join(manifest_rows).encode("utf-8")).hexdigest()
    return reference, split_counts, f"{data_dir.name}:{digest[:16]}"


def make_run_id(experiment: str, seed: int, backbone: str) -> str:
    return f"taskB17__{experiment}__seed_{seed}__{backbone}".replace(".", "p")


def discover_checkpoints(experiment_dir: Path) -> pd.DataFrame:
    records = []
    for checkpoint in sorted(experiment_dir.rglob("*_best.pth")):
        parts = checkpoint.relative_to(experiment_dir).parts
        if parts[0] == "main":
            experiment, seed, backbone = "main", int(parts[1].removeprefix("seed_")), parts[2]
        else:
            experiment, seed, backbone = parts[0], 42, parts[1]
        records.append({
            "run_id": make_run_id(experiment, seed, backbone),
            "experiment": experiment, "seed": seed, "backbone": backbone,
            "checkpoint": checkpoint,
        })

    inventory = pd.DataFrame(records)
    expected = {
        *(f"main|{seed}|{backbone}" for seed in MAIN_SEEDS for backbone in BACKBONES),
        *(f"{experiment}|42|{backbone}" for experiment in ABLATIONS for backbone in BACKBONES),
    }
    actual = {f"{row.experiment}|{row.seed}|{row.backbone}" for row in inventory.itertuples(index=False)}
    assert len(inventory) == 30, f"Expected 30 checkpoints, got {len(inventory)}"
    assert inventory["run_id"].is_unique, "run_id must be unique"
    assert actual == expected, f"Checkpoint mismatch. Missing={sorted(expected - actual)}, extra={sorted(actual - expected)}"
    return inventory.sort_values(["experiment", "seed", "backbone"]).reset_index(drop=True)


GIT_COMMIT = current_git_commit(ROOT)
class_names, split_counts, split_version = validate_dataset(DATA_DIR)
checkpoint_inventory = discover_checkpoints(EXPERIMENT_DIR)
print(f"Dataset: {len(class_names)} classes, split={split_version}, counts={split_counts}")
print(f"Git commit: {GIT_COMMIT}")
print(f"Checkpoint inventory: {len(checkpoint_inventory)} runs")
checkpoint_inventory[["run_id", "experiment", "seed", "backbone"]]

Dataset: 17 classes, split=task-B-group-split:359be7f5379818d4, counts={'train': 3365, 'val': 903, 'test': 532}
Git commit: 75df6c061eca5a3d63cfac706d1b6e256e404342
Checkpoint inventory: 30 runs


,run_id,experiment,seed,backbone
0,taskB17__main__seed_42__convnext_tiny,main,42,convnext_tiny
1,taskB17__main__seed_42__efficientnet_b3,main,42,efficientnet_b3
2,taskB17__main__seed_42__mobilenetv3_large,main,42,mobilenetv3_large
3,taskB17__main__seed_42__resnet50,main,42,resnet50
4,taskB17__main__seed_42__swin_tiny,main,42,swin_tiny
5,taskB17__main__seed_123__convnext_tiny,main,123,convnext_tiny
6,taskB17__main__seed_123__efficientnet_b3,main,123,efficientnet_b3
7,taskB17__main__seed_123__mobilenetv3_large,main,123,mobilenetv3_large
8,taskB17__main__seed_123__resnet50,main,123,resnet50
9,taskB17__main__seed_123__swin_tiny,main,123,swin_tiny


In [5]:
transform = get_val_transforms(img_size=IMAGE_SIZE)
datasets = {
    split: FlatConiferDataset(DATA_DIR, split=split, transform=transform, task="taskB", classes=class_names)
    for split in ["val", "test"]
}
loader_kwargs = {
    "batch_size": DATA_LOADING_BATCH_SIZE, "shuffle": False,
    "num_workers": NUM_WORKERS, "pin_memory": DEVICE.type == "cuda",
}
if NUM_WORKERS > 0:
    loader_kwargs.update(prefetch_factor=PREFETCH_FACTOR, persistent_workers=False)
loaders = {
    split: DataLoader(dataset, **loader_kwargs)
    for split, dataset in datasets.items()
}


def amp_context():
    return torch.autocast(device_type="cuda", dtype=torch.float16) if USE_AMP else nullcontext()


def prepare_images(images: torch.Tensor) -> torch.Tensor:
    if DEVICE.type != "cuda":
        return images.to(DEVICE)
    return images.to(
        device=DEVICE, dtype=torch.float16 if USE_AMP else torch.float32,
        non_blocking=True, memory_format=torch.channels_last,
    )


def materialize_eval_data(split: str, loader: DataLoader) -> dict:
    cache_dtype = torch.float16 if USE_AMP else torch.float32
    image_batches, target_batches = [], []
    for images, targets in loader:
        image_batches.append(images.to(dtype=cache_dtype))
        target_batches.append(targets)
    images = torch.cat(image_batches)
    targets = torch.cat(target_batches)
    cache_location = "cpu"

    if CACHE_EVAL_ON_GPU:
        required_bytes = images.numel() * images.element_size()
        free_bytes, _ = torch.cuda.mem_get_info()
        reserve_bytes = int(GPU_CACHE_RESERVE_GB * 2**30)
        if required_bytes + reserve_bytes < free_bytes:
            try:
                images = images.to(
                    device=DEVICE, non_blocking=True, memory_format=torch.channels_last
                )
                torch.cuda.synchronize()
                cache_location = "cuda"
            except RuntimeError as exc:
                if "out of memory" not in str(exc).lower():
                    raise
                torch.cuda.empty_cache()

    if cache_location == "cpu" and DEVICE.type == "cuda":
        try:
            images = images.pin_memory()
        except RuntimeError:
            pass

    cache_size_gb = images.numel() * images.element_size() / 2**30
    print(f"Cached {split}: {len(targets)} images | {cache_size_gb:.2f} GiB | {cache_location}")
    return {"images": images, "targets": targets, "location": cache_location}


eval_cache = {split: materialize_eval_data(split, loader) for split, loader in loaders.items()}
del loaders
gc.collect()


def load_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding="utf-8")) if path.exists() else {}


def load_checkpoint(path: Path):
    try:
        payload = torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        payload = torch.load(path, map_location="cpu")
    return payload, payload.get("model_state_dict", payload)


def checkpoint_metadata(payload: dict) -> dict:
    metrics = payload.get("metrics", {})
    best_epoch = payload.get("best_epoch", payload.get("epoch", -1))
    return {
        "checkpoint_best_val_acc": float(payload.get("best_val_acc", metrics.get("val_acc", np.nan))),
        "checkpoint_best_epoch": int(best_epoch) + 1,
        "checkpoint_saved_epoch": int(payload.get("epoch", -1)) + 1,
    }


def run_metadata(record, payload: dict) -> dict:
    run_config_path = record.checkpoint.parent / "run_config.json"
    training_metrics_path = record.checkpoint.parent / "test_metrics.json"
    run_config, training_metrics = load_json(run_config_path), load_json(training_metrics_path)
    defaults = EXPERIMENT_DEFAULTS[record.experiment]
    use_90rotation = bool(run_config.get("use_90rotation", defaults["use_90rotation"]))
    augmentation = (
        "resize224|hflip0.5|vflip0.5|random_90|color_jitter0.2" if use_90rotation
        else "resize224|hflip0.5|vflip0.5|random_rotation30|color_jitter0.2"
    )
    checkpoint_time = datetime.fromtimestamp(record.checkpoint.stat().st_mtime, tz=timezone.utc).isoformat()
    configured_run_date = run_config.get("run_date")
    prediction_path = PREDICTION_DIR / f"{record.run_id}-predictions.csv"
    confusion_matrix_paths = {
        split: record.checkpoint.parent / f"confusion_matrix_{split}.png"
        for split in ["val", "test"]
    }
    return {
        "run_id": record.run_id,
        "run_date": configured_run_date or checkpoint_time,
        "run_date_source": "run_config" if configured_run_date else "checkpoint_mtime_utc",
        "notebook": run_config.get("notebook", TRAINING_NOTEBOOKS[record.backbone]),
        "git_commit": run_config.get("git_commit", GIT_COMMIT),
        "git_commit_source": "run_config" if run_config.get("git_commit") else "evaluation_checkout",
        "split_version": split_version,
        "num_classes": len(class_names),
        "n_train": split_counts["train"], "n_val": split_counts["val"], "n_test": split_counts["test"],
        "seed": int(record.seed),
        "lr": float(run_config.get("lr", run_config.get("lr_head", 1e-4))),
        "lr_backbone": float(run_config.get("lr_backbone", 1e-5)),
        "epochs": int(run_config.get("epochs", 50)),
        "epochs_trained": int(training_metrics.get("epochs_trained", int(payload.get("epoch", -1)) + 1)),
        "batch": int(run_config.get("batch", run_config.get("batch_size", BACKBONE_BATCH_SIZES[record.backbone]))),
        "image_size": int(run_config.get("image_size", run_config.get("img_size", IMAGE_SIZE))),
        "label_smoothing": float(run_config.get("label_smoothing", defaults["label_smoothing"])),
        "augmentation": run_config.get("augmentation", augmentation),
        "experiment": record.experiment, "backbone": record.backbone,
        "checkpoint_path": relative_to_root(record.checkpoint),
        "prediction_path": relative_to_root(prediction_path),
        **{
            f"{split}_confusion_matrix_path": relative_to_root(path)
            for split, path in confusion_matrix_paths.items()
        },
        "run_config_path": relative_to_root(run_config_path) if run_config_path.exists() else "",
        "evaluation_notebook": EVALUATION_NOTEBOOK,
        **checkpoint_metadata(payload),
    }


preflight_record = checkpoint_inventory.iloc[0]
preflight_payload, preflight_state = load_checkpoint(preflight_record.checkpoint)
preflight_model = create_model(preflight_record.backbone, num_classes=len(class_names), pretrained=False).to(DEVICE)
preflight_model.load_state_dict(preflight_state, strict=True)
preflight_model.requires_grad_(False).eval()
if DEVICE.type == "cuda":
    preflight_model = preflight_model.to(memory_format=torch.channels_last)
preflight_images = prepare_images(eval_cache["val"]["images"][:MIN_EVAL_BATCH_SIZE])
with torch.inference_mode(), amp_context():
    preflight_logits = preflight_model(preflight_images)
assert preflight_logits.shape == (len(preflight_images), len(class_names))
print(f"Preflight passed: logits={tuple(preflight_logits.shape)}")
del preflight_model, preflight_payload, preflight_state, preflight_images, preflight_logits
gc.collect()
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

Cached val: 903 images | 0.25 GiB | cuda
Cached test: 532 images | 0.15 GiB | cuda
Preflight passed: logits=(16, 17)


In [6]:
METRIC_COLUMNS = [
    "log_loss", "accuracy", "balanced_accuracy",
    "precision_macro", "recall_macro", "f1_macro",
    "precision_micro", "recall_micro", "f1_micro",
    "precision_weighted", "recall_weighted", "f1_weighted",
    "matthews_corrcoef", "cohen_kappa", "top3_accuracy",
]


@torch.inference_mode()
def infer(model: torch.nn.Module, cache: dict, batch_size: int):
    labels = cache["targets"].numpy()
    predictions, probabilities = [], []
    for start in range(0, len(labels), batch_size):
        images = prepare_images(cache["images"][start:start + batch_size])
        with amp_context():
            logits = model(images)
        probs = torch.softmax(logits.float(), dim=1)
        predictions.append(probs.argmax(dim=1).cpu().numpy())
        probabilities.append(probs.cpu().numpy())
    return labels, np.concatenate(predictions), np.concatenate(probabilities)


def infer_with_oom_fallback(model: torch.nn.Module, cache: dict, requested_batch_size: int):
    batch_size = min(requested_batch_size, len(cache["targets"]))
    initial_batch_size = batch_size
    while True:
        try:
            outputs = infer(model, cache, batch_size)
            return (*outputs, batch_size, batch_size < initial_batch_size)
        except RuntimeError as exc:
            is_cuda_oom = DEVICE.type == "cuda" and "out of memory" in str(exc).lower()
            if not is_cuda_oom or batch_size <= MIN_EVAL_BATCH_SIZE:
                raise
            next_batch_size = max(MIN_EVAL_BATCH_SIZE, batch_size // 2)
            print(f"CUDA OOM at batch={batch_size}; retrying with batch={next_batch_size}")
            batch_size = next_batch_size
            gc.collect()
            torch.cuda.empty_cache()


def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray, y_prob: np.ndarray):
    label_ids = np.arange(len(class_names))
    aggregate = {
        "log_loss": float(log_loss(y_true, y_prob, labels=label_ids)),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "matthews_corrcoef": float(matthews_corrcoef(y_true, y_pred)),
        "cohen_kappa": float(cohen_kappa_score(y_true, y_pred)),
        "top3_accuracy": float(top_k_accuracy_score(y_true, y_prob, k=3, labels=label_ids)),
    }
    for average in ["macro", "micro", "weighted"]:
        precision, recall, f1, _ = precision_recall_fscore_support(
            y_true, y_pred, labels=label_ids, average=average, zero_division=0,
        )
        aggregate[f"precision_{average}"] = float(precision)
        aggregate[f"recall_{average}"] = float(recall)
        aggregate[f"f1_{average}"] = float(f1)

    precision, recall, f1, support = precision_recall_fscore_support(
        y_true, y_pred, labels=label_ids, average=None, zero_division=0
    )
    per_class = pd.DataFrame({
        "class_index": label_ids, "class_name": class_names,
        "precision": precision, "recall": recall, "f1": f1,
        "support": support.astype(int),
    })
    return aggregate, per_class, confusion_matrix(y_true, y_pred, labels=label_ids)


def prediction_table(dataset, y_true, y_pred, y_prob) -> pd.DataFrame:
    image_paths = [relative_to_root(path) for path, _ in dataset.samples]
    top3_indices = np.argsort(y_prob, axis=1)[:, -3:][:, ::-1]
    return pd.DataFrame({
        "image_path": image_paths, "true_index": y_true,
        "true_class": [class_names[index] for index in y_true],
        "predicted_index": y_pred,
        "predicted_class": [class_names[index] for index in y_pred],
        "confidence": y_prob.max(axis=1), "correct": y_true == y_pred,
        "top3_classes": [" | ".join(class_names[index] for index in indices) for indices in top3_indices],
    })

In [7]:
split_metric_rows, registry_rows = [], []
per_class_frames, prediction_frames, confusion_matrices = [], [], {}
backbone_eval_batch_sizes = {}

for run_index, record in enumerate(checkpoint_inventory.itertuples(index=False), start=1):
    payload, state_dict = load_checkpoint(record.checkpoint)
    model = create_model(record.backbone, num_classes=len(class_names), pretrained=False)
    model.load_state_dict(state_dict, strict=True)
    model = model.to(DEVICE).requires_grad_(False).eval()
    if DEVICE.type == "cuda":
        model = model.to(memory_format=torch.channels_last)
    registry_row = run_metadata(record, payload)
    del payload, state_dict
    eval_batch_size = backbone_eval_batch_sizes.get(record.backbone, MAX_EVAL_BATCH_SIZE)

    for split in ["val", "test"]:
        y_true, y_pred, y_prob, used_batch_size, reduced_batch = infer_with_oom_fallback(
            model, eval_cache[split], eval_batch_size
        )
        if split == "val" or reduced_batch:
            eval_batch_size = used_batch_size
            backbone_eval_batch_sizes[record.backbone] = eval_batch_size
        aggregate, per_class, matrix = compute_metrics(y_true, y_pred, y_prob)
        confusion_key = f"{record.run_id}__{split}"
        common = {
            "run_id": record.run_id, "experiment": record.experiment,
            "seed": int(record.seed), "backbone": record.backbone, "split": split,
            "num_classes": len(class_names), "num_images": len(y_true),
            "evaluation_batch_size": used_batch_size,
            "checkpoint_path": registry_row["checkpoint_path"],
            "prediction_path": registry_row["prediction_path"],
            "confusion_matrix_key": confusion_key,
            "confusion_matrix_path": registry_row[f"{split}_confusion_matrix_path"],
        }
        split_metric_rows.append({**common, **aggregate})
        per_class_frames.append(per_class.assign(**common))
        prediction_frames.append(prediction_table(datasets[split], y_true, y_pred, y_prob).assign(**common))
        confusion_matrices[confusion_key] = matrix
        registry_row.update({f"{split}_{metric}": aggregate[metric] for metric in METRIC_COLUMNS})
        print(
            f"[{run_index:02d}/{len(checkpoint_inventory)}] {record.experiment:<27s} | "
            f"seed={record.seed:<3d} | {record.backbone:<20s} | "
            f"{split:<4s} | Acc={aggregate['accuracy'] * 100:6.2f}% | F1={aggregate['f1_macro'] * 100:6.2f}%"
            f" | batch={used_batch_size}"
        )

    registry_row.update({
        "evaluation_batch_size": eval_batch_size,
        "evaluation_precision": "amp_fp16" if USE_AMP else "fp32",
        "evaluation_cache": "|".join(f"{split}:{eval_cache[split]['location']}" for split in ["val", "test"]),
    })
    registry_rows.append(registry_row)
    del model
    if run_index % 5 == 0:
        gc.collect()

run_metrics = pd.DataFrame(split_metric_rows)
registry = pd.DataFrame(registry_rows)
per_class_metrics = pd.concat(per_class_frames, ignore_index=True)
predictions = pd.concat(prediction_frames, ignore_index=True)
REQUIRED_REGISTRY_COLUMNS = [
    "run_id", "run_date", "notebook", "git_commit", "split_version",
    "num_classes", "n_train", "n_val", "n_test", "seed", "lr",
    "epochs", "batch", "image_size", "label_smoothing", "augmentation",
    "checkpoint_path", "prediction_path",
    *(f"{split}_confusion_matrix_path" for split in ["val", "test"]),
    *(f"{split}_{metric}" for split in ["val", "test"] for metric in METRIC_COLUMNS),
]
missing_columns = sorted(set(REQUIRED_REGISTRY_COLUMNS) - set(registry.columns))
assert not missing_columns, f"Registry missing columns: {missing_columns}"
assert len(registry) == 30 and registry["run_id"].is_unique
assert len(run_metrics) == 60
registry = registry[REQUIRED_REGISTRY_COLUMNS + [c for c in registry.columns if c not in REQUIRED_REGISTRY_COLUMNS]]
registry

[01/30] main                        | seed=42  | convnext_tiny        | val  | Acc= 99.89% | F1= 99.88% | batch=903
[01/30] main                        | seed=42  | convnext_tiny        | test | Acc= 99.81% | F1= 99.83% | batch=532
[02/30] main                        | seed=42  | efficientnet_b3      | val  | Acc= 96.12% | F1= 96.14% | batch=903
[02/30] main                        | seed=42  | efficientnet_b3      | test | Acc= 97.37% | F1= 97.40% | batch=532
[03/30] main                        | seed=42  | mobilenetv3_large    | val  | Acc= 94.91% | F1= 94.72% | batch=903
[03/30] main                        | seed=42  | mobilenetv3_large    | test | Acc= 98.68% | F1= 98.79% | batch=532
[04/30] main                        | seed=42  | resnet50             | val  | Acc= 90.48% | F1= 89.51% | batch=903
[04/30] main                        | seed=42  | resnet50             | test | Acc= 93.80% | F1= 93.94% | batch=532
[05/30] main                        | seed=42  | swin_tiny            | 

,run_id,run_date,notebook,git_commit,split_version,num_classes,n_train,n_val,n_test,seed,...,experiment,backbone,run_config_path,evaluation_notebook,checkpoint_best_val_acc,checkpoint_best_epoch,checkpoint_saved_epoch,evaluation_batch_size,evaluation_precision,evaluation_cache
0,taskB17__main__seed_42__convnext_tiny,2026-08-18T14:50:41.401972+00:00,notebooks/phase 1/task-B-17-new/04_train_convn...,75df6c061eca5a3d63cfac706d1b6e256e404342,task-B-group-split:359be7f5379818d4,17,3365,903,532,42,...,main,convnext_tiny,data/experiments/Task-B-new/final/main/seed_42...,notebooks/phase 1/task-B-17-new/06_evaluate_an...,99.889258,14,14,903,amp_fp16,val:cuda|test:cuda
1,taskB17__main__seed_42__efficientnet_b3,2026-08-18T16:12:20.028914+00:00,notebooks/phase 1/task-B-17-new/02_train_effic...,75df6c061eca5a3d63cfac706d1b6e256e404342,task-B-group-split:359be7f5379818d4,17,3365,903,532,42,...,main,efficientnet_b3,data/experiments/Task-B-new/final/main/seed_42...,notebooks/phase 1/task-B-17-new/06_evaluate_an...,96.124031,43,43,903,amp_fp16,val:cuda|test:cuda
2,taskB17__main__seed_42__mobilenetv3_large,2026-08-18T15:52:26.606598+00:00,notebooks/phase 1/task-B-17-new/03_train_mobil...,75df6c061eca5a3d63cfac706d1b6e256e404342,task-B-group-split:359be7f5379818d4,17,3365,903,532,42,...,main,mobilenetv3_large,data/experiments/Task-B-new/final/main/seed_42...,notebooks/phase 1/task-B-17-new/06_evaluate_an...,94.905869,43,43,903,amp_fp16,val:cuda|test:cuda
3,taskB17__main__seed_42__resnet50,2026-08-18T16:19:12.762547+00:00,notebooks/phase 1/task-B-17-new/01_train_resne...,75df6c061eca5a3d63cfac706d1b6e256e404342,task-B-group-split:359be7f5379818d4,17,3365,903,532,42,...,main,resnet50,data/experiments/Task-B-new/final/main/seed_42...,notebooks/phase 1/task-B-17-new/06_evaluate_an...,90.476190,47,47,903,amp_fp16,val:cuda|test:cuda
4,taskB17__main__seed_42__swin_tiny,2026-08-18T14:43:49.952933+00:00,notebooks/phase 1/task-B-17-new/05_train_swin_...,75df6c061eca5a3d63cfac706d1b6e256e404342,task-B-group-split:359be7f5379818d4,17,3365,903,532,42,...,main,swin_tiny,data/experiments/Task-B-new/final/main/seed_42...,notebooks/phase 1/task-B-17-new/06_evaluate_an...,99.557032,10,10,903,amp_fp16,val:cuda|test:cuda
5,taskB17__main__seed_123__convnext_tiny,2026-08-18T15:45:41.923428+00:00,notebooks/phase 1/task-B-17-new/04_train_convn...,75df6c061eca5a3d63cfac706d1b6e256e404342,task-B-group-split:359be7f5379818d4,17,3365,903,532,123,...,main,convnext_tiny,data/experiments/Task-B-new/final/main/seed_12...,notebooks/phase 1/task-B-17-new/06_evaluate_an...,99.889258,15,15,903,amp_fp16,val:cuda|test:cuda
6,taskB17__main__seed_123__efficientnet_b3,2026-08-18T18:03:34.634391+00:00,notebooks/phase 1/task-B-17-new/02_train_effic...,75df6c061eca5a3d63cfac706d1b6e256e404342,task-B-group-split:359be7f5379818d4,17,3365,903,532,123,...,main,efficientnet_b3,data/experiments/Task-B-new/final/main/seed_12...,notebooks/phase 1/task-B-17-new/06_evaluate_an...,96.456257,37,37,903,amp_fp16,val:cuda|test:cuda
7,taskB17__main__seed_123__mobilenetv3_large,2026-08-18T18:00:11.762113+00:00,notebooks/phase 1/task-B-17-new/03_train_mobil...,75df6c061eca5a3d63cfac706d1b6e256e404342,task-B-group-split:359be7f5379818d4,17,3365,903,532,123,...,main,mobilenetv3_large,data/experiments/Task-B-new/final/main/seed_12...,notebooks/phase 1/task-B-17-new/06_evaluate_an...,95.570321,50,50,903,amp_fp16,val:cuda|test:cuda
8,taskB17__main__seed_123__resnet50,2026-08-18T18:24:54.452924+00:00,notebooks/phase 1/task-B-17-new/01_train_resne...,75df6c061eca5a3d63cfac706d1b6e256e404342,task-B-group-split:359be7f5379818d4,17,3365,903,532,123,...,main,resnet50,data/experiments/Task-B-new/final/main/seed_12...,notebooks/phase 1/task-B-17-new/06_evaluate_an...,90.254707,45,45,903,amp_fp16,val:cuda|test:cuda
9,taskB17__main__seed_123__swin_tiny,2026-08-18T16:05:13.111952+00:00,notebooks/phase 1/task-B-17-new/05_train_swin_...,75df6c061eca5a3d63cfac706d1b6e256e404342,task-B-group-split:359be7f5379818d4,17,3365,903

In [8]:
main_test = run_metrics.query("experiment == 'main' and split == 'test'")
summary_rows = []
for backbone, group in main_test.groupby("backbone", sort=False):
    summary = {
        "backbone": backbone, "runs": len(group),
        "source_run_ids": "|".join(sorted(group["run_id"])),
    }
    for metric in METRIC_COLUMNS:
        summary[f"{metric}_mean"] = group[metric].mean()
        summary[f"{metric}_std"] = group[metric].std(ddof=0)
    summary_rows.append(summary)
main_test_summary = pd.DataFrame(summary_rows)

baseline = main_test.query("seed == 42")[["run_id", "backbone", *METRIC_COLUMNS]].rename(
    columns={"run_id": "baseline_run_id", **{m: f"main_seed42_{m}" for m in METRIC_COLUMNS}}
)
ablation_comparison = run_metrics.query("experiment != 'main' and split == 'test'").merge(
    baseline, on="backbone", validate="many_to_one"
)
for metric in METRIC_COLUMNS:
    ablation_comparison[f"{metric}_delta_vs_main"] = (
        ablation_comparison[metric] - ablation_comparison[f"main_seed42_{metric}"]
    )

inventory_to_save = checkpoint_inventory.assign(
    checkpoint_path=checkpoint_inventory["checkpoint"].map(relative_to_root)
).drop(columns="checkpoint")
valid_run_ids = set(registry["run_id"])
for table_name, table in {
    "checkpoint_inventory": inventory_to_save, "split_metrics": run_metrics,
    "per_class_metrics": per_class_metrics, "predictions": predictions,
}.items():
    assert "run_id" in table and set(table["run_id"]) <= valid_run_ids, f"Bad lineage: {table_name}"
for source_ids in main_test_summary["source_run_ids"]:
    assert set(source_ids.split("|")) <= valid_run_ids
assert set(ablation_comparison["run_id"]) <= valid_run_ids
assert set(ablation_comparison["baseline_run_id"]) <= valid_run_ids

lineage_rows = [
    {"report_file": "split_metrics.csv", "report_row": f"{row.run_id}|{row.split}", "run_id": row.run_id, "role": "source"}
    for row in run_metrics.itertuples(index=False)
]
for row in main_test_summary.itertuples(index=False):
    lineage_rows.extend(
        {"report_file": "main_test_summary.csv", "report_row": row.backbone, "run_id": source_id, "role": "source"}
        for source_id in row.source_run_ids.split("|")
    )
for row in ablation_comparison.itertuples(index=False):
    lineage_rows.extend([
        {"report_file": "ablation_vs_main_seed42.csv", "report_row": row.run_id, "run_id": row.run_id, "role": "candidate"},
        {"report_file": "ablation_vs_main_seed42.csv", "report_row": row.run_id, "run_id": row.baseline_run_id, "role": "baseline"},
    ])
report_lineage = pd.DataFrame(lineage_rows)
assert set(report_lineage["run_id"]) <= valid_run_ids

metadata = {
    "registry_created_at_utc": REGISTRY_CREATED_AT.isoformat(),
    "registry_file": relative_to_root(REGISTRY_PATH),
    "evaluation_notebook": EVALUATION_NOTEBOOK, "git_commit": GIT_COMMIT,
    "split_version": split_version,
    "split_version_method": "sha256(relative_image_path + image_sha256)[:16]",
    "device_type": DEVICE.type,
    "device_name": torch.cuda.get_device_name(DEVICE) if DEVICE.type == "cuda" else platform.processor() or platform.machine(),
    "python_version": platform.python_version(), "torch_version": torch.__version__,
    "inference_optimization": {
        "amp_fp16": USE_AMP, "tf32": DEVICE.type == "cuda", "channels_last": DEVICE.type == "cuda",
        "eval_cache_location": {split: eval_cache[split]["location"] for split in ["val", "test"]},
        "backbone_eval_batch_sizes": backbone_eval_batch_sizes,
        "max_eval_batch_size": MAX_EVAL_BATCH_SIZE, "gpu_cache_reserve_gb": GPU_CACHE_RESERVE_GB,
    },
    "classes": class_names, "excluded_classes": sorted(EXCLUDED_CLASSES),
    "split_counts": split_counts, "registered_runs": len(registry),
    "evaluated_splits": ["val", "test"], "metric_columns": METRIC_COLUMNS,
    "confusion_matrix_artifacts": {"format": "png", "splits": ["val", "test"]},
    "lineage_rule": "run_id/source_run_ids/baseline_run_id values resolve to registry rows",
}

if RUN_OUTPUT_DIR.exists():
    raise FileExistsError(f"Refusing to overwrite an existing registry run: {RUN_OUTPUT_DIR}")
PREDICTION_DIR.mkdir(parents=True, exist_ok=False)
for metric_row in run_metrics.itertuples(index=False):
    save_confusion_matrix(
        cm=confusion_matrices[metric_row.confusion_matrix_key],
        class_names=class_names,
        output_path=ROOT / metric_row.confusion_matrix_path,
        title=f"{metric_row.run_id} | {metric_row.split}",
    )
registry.to_csv(REGISTRY_PATH, index=False)
inventory_to_save.to_csv(RUN_OUTPUT_DIR / "checkpoint_inventory.csv", index=False)
run_metrics.to_csv(RUN_OUTPUT_DIR / "split_metrics.csv", index=False)
run_metrics.to_json(RUN_OUTPUT_DIR / "split_metrics.json", orient="records", indent=2, force_ascii=False)
per_class_metrics.to_csv(RUN_OUTPUT_DIR / "per_class_metrics.csv", index=False)
predictions.to_csv(RUN_OUTPUT_DIR / "predictions.csv", index=False)
registry_by_id = registry.set_index("run_id")
for run_id, frame in predictions.groupby("run_id", sort=False):
    frame.to_csv(ROOT / registry_by_id.at[run_id, "prediction_path"], index=False)
main_test_summary.to_csv(RUN_OUTPUT_DIR / "main_test_summary.csv", index=False)
ablation_comparison.to_csv(RUN_OUTPUT_DIR / "ablation_vs_main_seed42.csv", index=False)
report_lineage.to_csv(RUN_OUTPUT_DIR / "report_lineage.csv", index=False)
np.savez_compressed(RUN_OUTPUT_DIR / "confusion_matrices.npz", **confusion_matrices)
(RUN_OUTPUT_DIR / "metadata.json").write_text(
    json.dumps(metadata, indent=2, ensure_ascii=False), encoding="utf-8"
)
assert all((ROOT / path).exists() for path in registry["checkpoint_path"])
assert all((ROOT / path).exists() for path in registry["prediction_path"])
assert all((ROOT / path).exists() for path in run_metrics["confusion_matrix_path"])
print(f"Saved registry and reports to: {RUN_OUTPUT_DIR}")
print(f"Registry rows: {len(registry)} -> {REGISTRY_PATH.name}")
print(f"Confusion matrices: {len(run_metrics)} PNG files saved beside checkpoints")
main_test_summary[[
    "backbone", "source_run_ids", "accuracy_mean", "accuracy_std",
    "f1_macro_mean", "f1_macro_std",
]].assign(
    accuracy_mean=lambda x: x.accuracy_mean * 100, accuracy_std=lambda x: x.accuracy_std * 100,
    f1_macro_mean=lambda x: x.f1_macro_mean * 100, f1_macro_std=lambda x: x.f1_macro_std * 100,
).round(3)

Saved registry and reports to: /workspace/woodbench/docs/reports/experiments/phase_1b/final_b_new_recomputed/20260819T043733Z
Registry rows: 30 -> task-b-17class-20260819T043733Z-registry.csv
Confusion matrices: 60 PNG files saved beside checkpoints


,backbone,source_run_ids,accuracy_mean,accuracy_std,f1_macro_mean,f1_macro_std
0,convnext_tiny,taskB17__main__seed_123__convnext_tiny|taskB17...,99.561,0.354,99.596,0.326
1,efficientnet_b3,taskB17__main__seed_123__efficientnet_b3|taskB...,97.807,0.319,97.844,0.322
2,mobilenetv3_large,taskB17__main__seed_123__mobilenetv3_large|tas...,98.371,0.319,98.448,0.345
3,resnet50,taskB17__main__seed_123__resnet50|taskB17__mai...,92.356,1.022,92.569,0.982
4,swin_tiny,taskB17__main__seed_123__swin_tiny|taskB17__ma...,99.123,0.581,99.196,0.533
